# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    #api_base="https://openai.vocareum.com/v1"
)

In [7]:
# TODO: Create a collection
# Choose any name you want
try:
    chroma_client.delete_collection("udaplay")
except Exception:
    pass  # collection didn't exist (or older client behavior)

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

In [8]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)
from lib.vector_db import VectorStoreManager
# We'll use the VectorStoreManager, but override its client to be persistent.
manager = VectorStoreManager(openai_api_key=os.getenv("OPENAI_API_KEY"))
manager.chroma_client = chroma_client
# manager.embedding_function = embedding_fn
store = manager.get_or_create_store("udaplay")

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [10]:
# ---------- Build a DataFrame from the JSON files ----------
import pandas as pd

dict_list = []
for file in sorted(os.listdir(data_dir)):
    if not file.endswith(".json"):
        continue
    with open(os.path.join(data_dir, file), "r", encoding="utf-8") as f:
        new_json = json.loads(f.read())
        dict_list.append(new_json)

df = pd.DataFrame(dict_list)

# Make YearOfRelease numeric (clean formatting)
if "YearOfRelease" in df.columns:
    df["YearOfRelease"] = pd.to_numeric(df["YearOfRelease"], errors="coerce")

# Display / verify formatting
df[["Name", "Platform", "Genre", "Publisher", "Description", "YearOfRelease"]].head(15)

,Name,Platform,Genre,Publisher,Description,YearOfRelease
0,Gran Turismo,PlayStation 1,Racing,Sony Computer Entertainment,A realistic racing simulator featuring a wide ...,1997
1,Grand Theft Auto: San Andreas,PlayStation 2,Action-adventure,Rockstar Games,An expansive open-world game set in the fictio...,2004
2,Gran Turismo 5,PlayStation 3,Racing,Sony Computer Entertainment,A comprehensive racing simulator featuring a v...,2010
3,Marvel's Spider-Man,PlayStation 4,Action-adventure,Sony Interactive Entertainment,An open-world superhero game that lets players...,2018
4,Marvel's Spider-Man 2,PlayStation 5,Action-adventure,Sony Interactive Entertainment,"The sequel to the acclaimed Spider-Man game, f...",2023
5,Pokémon Gold and Silver,Game Boy Color,Role-playing,Nintendo,Second-generation Pokémon games introducing ne...,1999
6,Pokémon Ruby and Sapphire,Game Boy Advance,Role-playing,Nintendo,Third-generation Pokémon games set in the Hoen...,2002
7,Super Mario World,Super Nintendo Entertainment System (SNES),Platformer,Nintendo,A classic platformer where Mario embarks on a ...,1990
8,Super Mario 64,Nintendo 64,Platformer,Nintendo,A groundbreaking 3D platformer that set new st...,1996
9,Super Smash Bros. Melee,GameCube,Fighting,Nintendo,A crossover fighting game featuring characters...,2001


In [11]:
# ---------- Demonstrate semantic search queries ----------
def run_query(q: str, k: int = 5):
    results = collection.query(
        query_texts=[q],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    print("\n" + "=" * 80)
    print(f"QUERY: {q}")
    print("=" * 80)

    for i in range(len(results["ids"][0])):
        md = results["metadatas"][0][i]
        dist = results["distances"][0][i]
        doc = results["documents"][0][i]

        print(f"\nRank {i+1} | distance={dist:.4f}")
        print(f"Name: {md.get('Name')}")
        print(f"Platform: {md.get('Platform')}")
        print(f"Genre: {md.get('Genre')}")
        print(f"Publisher: {md.get('Publisher')}")
        print(f"Year: {md.get('YearOfRelease')}")
        print(f"Doc: {doc}")

# A few semantic queries that should work well
run_query("realistic racing simulator with lots of cars and tracks", k=3)   # expects Gran Turismo near top
run_query("sports mini-games using motion controls", k=3)                  # expects Wii Sports near top
run_query("open-world superhero game with Spider-Man", k=3)                # expects Marvel's Spider-Man near top
run_query("Nintendo kart racing game", k=3)                                # expects Mario Kart 8 Deluxe near top



QUERY: realistic racing simulator with lots of cars and tracks

Rank 1 | distance=0.2330
Name: Gran Turismo
Platform: PlayStation 1
Genre: Racing
Publisher: Sony Computer Entertainment
Year: 1997
Doc: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

Rank 2 | distance=0.2493
Name: Gran Turismo 5
Platform: PlayStation 3
Genre: Racing
Publisher: Sony Computer Entertainment
Year: 2010
Doc: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

Rank 3 | distance=0.3883
Name: Mario Kart 8 Deluxe
Platform: Nintendo Switch
Genre: Racing
Publisher: Nintendo
Year: 2017
Doc: [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.

QUERY: sports mini-games using motion controls

Rank 1 | distance=0.2785
N

In [12]:
print("Collection count:", collection.count())
sample = collection.get(limit=5, include=["metadatas", "documents"])
for md, doc in zip(sample["metadatas"], sample["documents"]):
    print(md.get("Name"), "|", md.get("Platform"), "|", md.get("YearOfRelease"))
    print(" ", doc)


Collection count: 15
Gran Turismo | PlayStation 1 | 1997
  [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
Grand Theft Auto: San Andreas | PlayStation 2 | 2004
  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.
Gran Turismo 5 | PlayStation 3 | 2010
  [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
Marvel's Spider-Man | PlayStation 4 | 2018
  [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.
Marvel's Spider-Man 2 | PlayStation 5 | 2023
  [PlayStation 5] Marvel's Spider-Man 2 (2023) - The sequel to the acclaimed Spider-Man game, featuring both Peter Parker 